# ClinicalTrials.gov API Data Collection
## Fetching 50,000+ trial records for outcome classification

In [9]:
import requests
import pandas as pd
import time
from pathlib import Path

## API Configuration

In [10]:
BASE_URL = "https://clinicaltrials.gov/api/v2/studies"
PAGE_SIZE = 1000          # max allowed per request
DELAY_BETWEEN_REQUESTS = 0.4  # seconds between pages, to be polite
MAX_RETRIES = 5
RETRY_BACKOFF = 2.0       # exponential backoff multiplier

# Modules to pull from each study record
FIELDS = [
    "protocolSection.identificationModule",
    "protocolSection.statusModule",
    "protocolSection.oversightModule",
    "protocolSection.designModule",
    "protocolSection.outcomesModule",
    "protocolSection.sponsorCollaboratorsModule",
    "protocolSection.armsInterventionsModule",
    "protocolSection.conditionsModule",
    "protocolSection.eligibilityModule",
    "protocolSection.contactsLocationsModule",
]

# Our three target outcome classes
TARGET_STATUSES = ["COMPLETED", "TERMINATED", "WITHDRAWN"]

## Helpers

In [11]:
def fetch_page(params: dict, retries: int = MAX_RETRIES) -> dict:
    """GET one page from the API with exponential backoff on failure."""
    for attempt in range(retries):
        try:
            resp = requests.get(BASE_URL, params=params, timeout=30)
            if resp.status_code == 200:
                return resp.json()
            elif resp.status_code == 429:
                wait = RETRY_BACKOFF ** (attempt + 2)
                print(f"  Rate-limited. Waiting {wait:.1f}s...")
                time.sleep(wait)
            else:
                print(f"  HTTP {resp.status_code} on attempt {attempt+1}. Retrying...")
                time.sleep(RETRY_BACKOFF ** attempt)
        except requests.exceptions.RequestException as e:
            print(f"  Request error on attempt {attempt+1}: {e}. Retrying...")
            time.sleep(RETRY_BACKOFF ** attempt)
    raise RuntimeError(f"Failed after {retries} retries.")


def safe_get(d: dict, *keys):
    """Traverse nested dict keys without raising on missing keys."""
    for key in keys:
        if not isinstance(d, dict):
            return None
        d = d.get(key)
    return d


def flatten_study(study: dict) -> dict:
    """Extract and flatten the fields we care about from one study record."""
    ps = study.get("protocolSection", {})
    status_mod    = ps.get("statusModule", {})
    design_mod    = ps.get("designModule", {})
    sponsor_mod   = ps.get("sponsorCollaboratorsModule", {})
    arms_mod      = ps.get("armsInterventionsModule", {})
    cond_mod      = ps.get("conditionsModule", {})
    elig_mod      = ps.get("eligibilityModule", {})
    id_mod        = ps.get("identificationModule", {})
    loc_mod       = ps.get("contactsLocationsModule", {})
    oversight_mod = ps.get("oversightModule", {})
    outcomes_mod  = ps.get("outcomesModule", {})

    design_info  = design_mod.get("designInfo", {})
    enroll_info  = design_mod.get("enrollmentInfo", {})
    masking_info = design_info.get("maskingInfo", {})

    interventions = arms_mod.get("interventions", []) or []
    collaborators = sponsor_mod.get("collaborators", []) or []
    locations     = loc_mod.get("locations", []) or []
    conditions    = cond_mod.get("conditions", []) or []
    phases        = design_mod.get("phases", []) or []

    # Drug/biological interventions only — needed for OpenFDA join
    drug_interventions = [
        i.get("name", "") for i in interventions
        if i.get("type", "").upper() in ("DRUG", "BIOLOGICAL")
    ]
    intervention_types = list({i.get("type", "") for i in interventions})

    primary_outcomes   = outcomes_mod.get("primaryOutcomes", []) or []
    secondary_outcomes = outcomes_mod.get("secondaryOutcomes", []) or []

    return {
        # Identifiers
        "nct_id":                    id_mod.get("nctId"),
        "brief_title":               id_mod.get("briefTitle"),

        # Target label + timeline
        "overall_status":            status_mod.get("overallStatus"),
        "start_date":                safe_get(status_mod, "startDateStruct", "date"),
        "primary_completion_date":   safe_get(status_mod, "primaryCompletionDateStruct", "date"),
        "completion_date":           safe_get(status_mod, "completionDateStruct", "date"),

        # Regulatory oversight
        "is_fda_regulated_drug":     oversight_mod.get("isFdaRegulatedDrug"),
        "is_fda_regulated_device":   oversight_mod.get("isFdaRegulatedDevice"),
        "has_dmc":                   oversight_mod.get("oversightHasDmc"),  # Data Monitoring Committee

        # Trial design
        "study_type":                design_mod.get("studyType"),
        "phases":                    "|".join(phases) if phases else None,
        "intervention_model":        design_info.get("interventionModel"),
        "primary_purpose":           design_info.get("primaryPurpose"),
        "masking":                   masking_info.get("masking"),
        "enrollment_count":          enroll_info.get("count"),
        "enrollment_type":           enroll_info.get("type"),

        # Outcome complexity
        "num_primary_outcomes":      len(primary_outcomes),
        "num_secondary_outcomes":    len(secondary_outcomes),

        # Sponsor
        "lead_sponsor_class":        safe_get(sponsor_mod, "leadSponsor", "class"),
        "num_collaborators":         len(collaborators),

        # Interventions
        "intervention_types":        "|".join(intervention_types) if intervention_types else None,
        "drug_names":                "|".join(drug_interventions) if drug_interventions else None,
        "is_drug_trial":             len(drug_interventions) > 0,

        # Conditions studied
        "conditions":                "|".join(conditions) if conditions else None,

        # Eligibility
        "minimum_age":               elig_mod.get("minimumAge"),
        "maximum_age":               elig_mod.get("maximumAge"),
        "sex":                       elig_mod.get("sex"),
        "healthy_volunteers":        elig_mod.get("healthyVolunteers"),

        "num_sites":                 len(locations),
    }

## Main Fetch Loop

We need >50k rows with non-null `drug_names` after dedup and critical-null drops.
~47% of trials are drug trials, so fetching ~115k total gives ~54k drug-trial rows with margin.
Caps are proportional to the observed class distribution.

In [12]:
def fetch_trials_for_status(status: str, max_rows: int) -> list[dict]:
    """Paginate through trials of a given status until max_rows is reached or pages run out."""
    records = []
    page_token = None
    page_num = 0

    while len(records) < max_rows:
        params = {
            "format": "json",
            "pageSize": PAGE_SIZE,
            "filter.overallStatus": status,
            "fields": ",".join(FIELDS),
        }
        if page_token:
            params["pageToken"] = page_token

        data = fetch_page(params)
        studies = data.get("studies", [])

        if not studies:
            break

        for study in studies:
            records.append(flatten_study(study))
            if len(records) >= max_rows:
                break

        page_token = data.get("nextPageToken")
        page_num += 1

        if page_num % 10 == 0:
            print(f"  [{status}] Page {page_num} — {len(records)} records so far")

        if not page_token:
            print(f"  [{status}] Exhausted all pages at {len(records)} records")
            break

        time.sleep(DELAY_BETWEEN_REQUESTS)

    return records


all_records = []

# Caps scaled so drug-trial rows (is_drug_trial=True) exceed 50k after cleaning
per_class_caps = {
    "COMPLETED":  70_000,
    "TERMINATED": 33_000,
    "WITHDRAWN":  12_000,
}

for status, cap in per_class_caps.items():
    print(f"\n=== Fetching {status} (cap={cap:,}) ===")
    records = fetch_trials_for_status(status, cap)
    print(f"  => {len(records):,} records fetched for {status}")
    all_records.extend(records)

print(f"\nTotal raw records fetched: {len(all_records):,}")


=== Fetching COMPLETED (cap=70,000) ===
  [COMPLETED] Page 10 — 10000 records so far
  [COMPLETED] Page 20 — 20000 records so far
  [COMPLETED] Page 30 — 30000 records so far
  [COMPLETED] Page 40 — 40000 records so far
  [COMPLETED] Page 50 — 50000 records so far
  [COMPLETED] Page 60 — 60000 records so far
  [COMPLETED] Page 70 — 70000 records so far
  => 70,000 records fetched for COMPLETED

=== Fetching TERMINATED (cap=33,000) ===
  [TERMINATED] Page 10 — 10000 records so far
  [TERMINATED] Page 20 — 20000 records so far
  [TERMINATED] Page 30 — 30000 records so far
  => 33,000 records fetched for TERMINATED

=== Fetching WITHDRAWN (cap=12,000) ===
  [WITHDRAWN] Page 10 — 10000 records so far
  => 12,000 records fetched for WITHDRAWN

Total raw records fetched: 115,000


## Build DataFrame and Initial Cleaning

In [13]:
df_raw = pd.DataFrame(all_records)
print(f"Raw shape: {df_raw.shape}")
print(f"\nClass distribution (raw):")
print(df_raw["overall_status"].value_counts())
print(f"\nRows with non-null drug_names: {df_raw['drug_names'].notna().sum():,}")

# Sanity check: no unexpected status values slipped through
assert df_raw["overall_status"].isin(TARGET_STATUSES).all(), "Unexpected status values found!"

# Drop exact duplicate NCT IDs (shouldn't happen, but be safe)
dupes = df_raw["nct_id"].duplicated().sum()
print(f"Duplicate nct_ids: {dupes}")
df_raw = df_raw.drop_duplicates(subset="nct_id")
print(f"Shape after dedup: {df_raw.shape}")

Raw shape: (115000, 29)

Class distribution (raw):
overall_status
COMPLETED     70000
TERMINATED    33000
WITHDRAWN     12000
Name: count, dtype: int64

Rows with non-null drug_names: 54,311
Duplicate nct_ids: 0
Shape after dedup: (115000, 29)


In [14]:
# Check how much data is missing per column
missing = df_raw.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(df_raw) * 100).round(1)
missing_report = pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct})
print("Missing value report:")
print(missing_report[missing_report["missing_count"] > 0].to_string())

# How many rows are missing at least one key feature
key_features = ["overall_status", "study_type", "phases", "lead_sponsor_class",
                "enrollment_count", "start_date"]
print(f"\nRows with at least one null in a key feature: {df_raw[key_features].isnull().any(axis=1).sum()}")
print(f"Rows with non-null drug_names: {df_raw['drug_names'].notna().sum():,}")

Missing value report:
                         missing_count  missing_pct
drug_names                       60689         52.8
is_fda_regulated_device          59412         51.7
is_fda_regulated_drug            59388         51.6
maximum_age                      56583         49.2
primary_purpose                  23348         20.3
intervention_model               23231         20.2
masking                          22866         19.9
phases                           21639         18.8
has_dmc                          18545         16.1
intervention_types                9414          8.2
minimum_age                       6827          5.9
primary_completion_date           4878          4.2
enrollment_type                   3572          3.1
completion_date                   3436          3.0
healthy_volunteers                1729          1.5
enrollment_count                  1131          1.0
start_date                         978          0.9
sex                                 74    

## Drop rows missing critical features

In [15]:
# Drop rows where the target or essential design features are missing — these are unusable for modeling
# Everything else (missing masking, ages, etc.) will be handled in the full preprocessing pass
must_have = ["overall_status", "study_type", "lead_sponsor_class", "start_date"]

before = len(df_raw)
df_clean = df_raw.dropna(subset=must_have)
after = len(df_clean)

print(f"Dropped {before - after:,} rows missing critical fields")
print(f"Remaining: {after:,} rows")
print(f"\nClass distribution:")
print(df_clean["overall_status"].value_counts())
print(f"\nRows with non-null drug_names: {df_clean['drug_names'].notna().sum():,}")

Dropped 978 rows missing critical fields
Remaining: 114,022 rows

Class distribution:
overall_status
COMPLETED     69541
TERMINATED    32878
WITHDRAWN     11603
Name: count, dtype: int64

Rows with non-null drug_names: 53,628


## Save to CSV

In [16]:
output_path = Path("clinical_trials_raw.csv")
df_clean.to_csv(output_path, index=False)
print(f"Saved {len(df_clean):,} rows x {df_clean.shape[1]} cols → {output_path.resolve()}")
print(f"\nColumn list:")
for col in df_clean.columns:
    print(f"  {col}")

Saved 114,022 rows x 29 cols → /Users/zheng/CIS2450/245-Project/clinical_trials_raw.csv

Column list:
  nct_id
  brief_title
  overall_status
  start_date
  primary_completion_date
  completion_date
  is_fda_regulated_drug
  is_fda_regulated_device
  has_dmc
  study_type
  phases
  intervention_model
  primary_purpose
  masking
  enrollment_count
  enrollment_type
  num_primary_outcomes
  num_secondary_outcomes
  lead_sponsor_class
  num_collaborators
  intervention_types
  drug_names
  is_drug_trial
  conditions
  minimum_age
  maximum_age
  sex
  healthy_volunteers
  num_sites
